In [1]:
import pandas as pd
import numpy as np


from sklearn.model_selection   import train_test_split
from sklearn.preprocessing    import LabelEncoder
from sklearn.metrics          import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, hamming_loss
from sklearn.multiclass import OneVsRestClassifier
from tensorflow import keras
from lightgbm import LGBMClassifier
from sklearn.preprocessing    import StandardScaler
from xgboost import XGBClassifier

1.) LOAD DATA INTO A SINGLE DATAFRAME AND INSPECT

In [3]:
# load product properties in a dataframe
product_props = pd.read_parquet('product_properties.parquet')[['sku','category','price']]

# Function to load the remaining files
def load_event(path, event_name):
    df = pd.read_parquet(path)
    df['event_type'] = event_name
    return df

# Load each event file
buys = load_event('product_buy.parquet', 'buy')
adds = load_event('add_to_cart.parquet', 'add_to_cart')
removes = load_event('remove_from_cart.parquet','remove_from_cart')
visits = load_event('page_visit.parquet', 'page_visit')
searches = load_event('search_query.parquet', 'search_query')



In [17]:
# Create a dtaframe with the events
all_events = pd.concat([buys, adds, removes, visits, searches],
                       axis=0, ignore_index=True)
all_events['timestamp'] = pd.to_datetime(all_events['timestamp'])

In [25]:
# Merge all_events and product properties
all_events = all_events.merge(
    product_props,
    on='sku',
    how='left'
)


In [31]:
removes

,client_id,timestamp,sku,event_type
0,10238779,2022-10-05 14:37:35,1475246,remove_from_cart
1,10238779,2022-10-05 14:37:40,1475246,remove_from_cart
2,10238779,2022-10-05 14:37:40,1475246,remove_from_cart
3,10238779,2022-10-05 14:37:40,1475246,remove_from_cart
4,17130676,2022-09-13 07:31:35,830819,remove_from_cart
...,...,...,...,...
1697886,10246048,2022-10-03 21:29:00,536615,remove_from_cart
1697887,10246048,2022-10-05 17:40:55,536615,remove_from_cart
1697888,10246048,2022-10-05 17:43:15,536615,remove_from_cart
1697889,10246048,2022-10-05 17:45:40,536615,remove_from_cart


In [ ]:
#all_events

In [1]:
#unique users
all_events['client_id'].nunique()

NameError: name 'all_events' is not defined

In [14]:

# Find the end of data and set cutoff 14 days earlier
data_end  = all_events['timestamp'].max()            
cutoff_dt = data_end - pd.Timedelta(days=14)          

#Display end date and cutoff date
print("Global data end:", data_end)
print("Cutoff date   :", cutoff_dt)



Global data end: 2022-10-10 23:59:59
Cutoff date   : 2022-09-26 23:59:59


In [16]:
# Create a pre cutoff dataframe named feat_df
feat_df  = all_events[ all_events['timestamp'] <=  cutoff_dt ].copy()

In [18]:
#Create a post cutoff dataframe named label_df
label_df = all_events[(all_events['timestamp'] >   cutoff_dt) &
                     (all_events['timestamp'] <=  data_end)].copy()

print("Feature window: ", feat_df['timestamp'].min(), "→", feat_df['timestamp'].max())
print("Label window:   ", label_df['timestamp'].min(), "→", label_df['timestamp'].max())

Feature window:  2022-05-23 00:10:00 → 2022-09-26 23:59:59
Label window:    2022-09-27 00:00:00 → 2022-10-10 23:59:59


# FIND USERS WHO HAVE MADE AT LEAST A PURCHASE IN LABEL WINDOW

In [20]:
#Users who bought something in the label window
label_buyers = label_df[label_df['event_type'] == 'buy']['client_id'].unique()

# Count the number of events in the feature window per user
event_counts = feat_df['client_id'].value_counts()
active_users = event_counts[event_counts >= 10].index

# Keep users who satisfy both conditions
eligible_users = np.intersect1d(label_buyers, active_users)

print("Users with purchase in label window:", len(label_buyers))
print("Users with ≥10 events in feature window:", len(active_users))
print("Eligible users (both conditions):", len(eligible_users))

# Filter both dataframes
feat_df = feat_df[feat_df['client_id'].isin(eligible_users)].copy()
label_df = label_df[label_df['client_id'].isin(eligible_users)].copy()


Users with purchase in label window: 95119
Users with ≥10 events in feature window: 2836680
Eligible users (both conditions): 32663


Find users with at least one buy in the post cutoff window

In [21]:
# find the "buy" events
label_buys = label_df[label_df['event_type'] == 'buy'].copy()

# Sort the buy events by client_id and timestamp 
label_buys = label_buys.sort_values(['client_id', 'timestamp'])

# Group purchases per user into a list of categories
label_categories = (
    label_buys
      .groupby('client_id')['category']
      .apply(list)
      .reset_index()
      .rename(columns={'category': 'label_categories'})
)

# Confirm the user count matches filtered label_df
print("Users with at least one label-window buy (eligible):", label_buys['client_id'].nunique())

# Explode categories for frequency analysis
exploded = label_categories.explode('label_categories')


Users with at least one label-window buy (eligible): 32663


FIND THE TOP 30 CATEGORIES IN TERMS OF MOST FREQUENT ONES 

In [26]:
buy_events = label_df[label_df['event_type'] == 'buy'].copy()
# Get distinct number of categories
unique_categories = buy_events['category'].nunique()

print(f"Distinct categories bought in label window: {unique_categories}")



Distinct categories bought in label window: 3723


In [28]:
# Find the top 30 categories that will be the subset from which the models will suggest
topK = 30
top_categories = label_buys['category'].value_counts().head(topK).index.tolist()

print(f"Selected top {topK} categories for prediction:", top_categories)


Selected top 30 categories for prediction: [3944.0, 5954.0, 3068.0, 2739.0, 4142.0, 3147.0, 5476.0, 3031.0, 1097.0, 4715.0, 6839.0, 5594.0, 3709.0, 5323.0, 6857.0, 2405.0, 2855.0, 3263.0, 4195.0, 2910.0, 1586.0, 4517.0, 3421.0, 4981.0, 1741.0, 3061.0, 5152.0, 3605.0, 1954.0, 4154.0]


# Feature Creation

In [30]:
#count the events per client id and event type
event_counts = feat_df.pivot_table(
    index='client_id',
    columns='event_type',
    values='sku',
    aggfunc='count',
    fill_value=0
).add_prefix('count_')


unique_cats = feat_df.groupby('client_id')['category'] \
                     .nunique() \
                     .rename('unique_categories')

In [32]:
# features regarding the general aggregated activity of each user

last_ts = feat_df.groupby('client_id')['timestamp'].max().rename('last_ts')
first_ts= feat_df.groupby('client_id')['timestamp'].min().rename('first_ts')

recency_days  = ((cutoff_dt - last_ts).dt.days).rename('recency_days')
active_days   = ((last_ts - first_ts).dt.days + 1).rename('active_days')
events_per_day= (feat_df.groupby('client_id').size() / active_days).rename('events_per_day')

# Create a new dataframe with the generated features
user_features = (
    event_counts
    .join(unique_cats)
    .join(recency_days)
    .join(active_days)
    .join(events_per_day)
    .reset_index()
)



Calculate the events per category for each user and enrich the features

In [34]:

cat_event_counts = (
    feat_df[feat_df['category'].isin(top_categories)]
    .groupby(['client_id', 'category', 'event_type'])
    .size()
    .reset_index(name='count')
)

# Create one column per category-event for each user
cat_event_features = (
    cat_event_counts
      .pivot(
         index='client_id',
         columns=['event_type','category'],
         values='count'
      )
      .fillna(0)
)

# Flatten multi-index columns 
cat_event_features.columns = [f'{event}_cat_{int(cat)}' for event, cat in cat_event_features.columns]
cat_event_features = cat_event_features.reset_index()

# Join with existing user_features
user_features = (
    user_features
      .merge(cat_event_features,
             on='client_id',
             how='left')
      .fillna(0)
)


print("user_features shape:", user_features.shape)
print(user_features.head())

user_features shape: (32663, 100)
   client_id  count_add_to_cart  count_buy  count_page_visit  \
0        616                  0          0                 0   
1       1326                 11          3                 0   
2       3425                  0          0                 0   
3       3526                  5          1                 0   
4       4384                  0          5                 0   

   count_remove_from_cart  count_search_query  unique_categories  \
0                       0                   0                  0   
1                       9                   0                  5   
2                       0                   0                  0   
3                       2                   0                  1   
4                       0                   0                  3   

   recency_days  active_days  events_per_day  ...  remove_from_cart_cat_1586  \
0            13           84        0.166667  ...                        0.0   
1           

Introduce ratios into the dataframe

In [38]:
# Initialize per-category ratio DataFrame
ratio_feats = pd.DataFrame({'client_id': user_features['client_id']})

# Collect all missing columns first
missing_cols = []

for cat in top_categories:
    add_col = f'add_to_cart_cat_{cat}'
    buy_col = f'buy_cat_{cat}'
    rem_col = f'remove_from_cart_cat_{cat}'
    view_col = f'page_visit_cat_{cat}'

    for col in [add_col, buy_col, rem_col, view_col]:
        if col not in user_features.columns:
            missing_cols.append(col)

# Create a DataFrame with 0s for missing columns
zero_fill = pd.DataFrame(0, index=user_features.index, columns=missing_cols)
user_features = pd.concat([user_features, zero_fill], axis=1)

# Compute ratios 
for cat in top_categories:
    add_col = f'add_to_cart_cat_{cat}'
    buy_col = f'buy_cat_{cat}'
    rem_col = f'remove_from_cart_cat_{cat}'
    view_col = f'page_visit_cat_{cat}'

    ratio_feats[f'cart_to_buy_ratio_cat_{cat}'] = user_features[buy_col] / (user_features[add_col] + 1)
    ratio_feats[f'remove_to_add_ratio_cat_{cat}'] = user_features[rem_col] / (user_features[add_col] + 1)
    ratio_feats[f'pageviews_per_buy_cat_{cat}'] = user_features[view_col] / (user_features[buy_col] + 1)

# Merge the ratios with the user_features df
user_features = user_features.merge(ratio_feats, on='client_id', how='left')


# Monetary features for each category-event pair

In [40]:
monetary_cat = (
    feat_df[feat_df['event_type'] == 'buy']
    .groupby(['client_id', 'category'])['price']
    .agg(['sum', 'mean', 'count'])
    .reset_index()
    .rename(columns={'sum': 'buy_total', 'mean': 'buy_avg_price', 'count': 'buy_count'})
)

# Keep only top categories
monetary_cat = monetary_cat[monetary_cat['category'].isin(top_categories)]

# Pivot 
monetary_cat_pivot = monetary_cat.pivot(index='client_id', columns='category')
monetary_cat_pivot.columns = [f"{stat}_cat_{int(cat)}" for stat, cat in monetary_cat_pivot.columns]
monetary_cat_pivot = monetary_cat_pivot.reset_index()


In [42]:
# Total buys per user
user_buys = (
    feat_df[feat_df['event_type'] == 'buy']
    .groupby('client_id')
    .size()
    .rename('total_buys')
)

# Buys per user-category
cat_buy_counts = (
    feat_df[(feat_df['event_type'] == 'buy') & (feat_df['category'].isin(top_categories))]
    .groupby(['client_id', 'category'])
    .size()
    .rename('cat_buy_count')
    .reset_index()
)

# Merge total buys
cat_buy_counts = cat_buy_counts.merge(user_buys, on='client_id')
cat_buy_counts['buy_ratio'] = cat_buy_counts['cat_buy_count'] / (cat_buy_counts['total_buys'] + 1)




# Temporal features per category 

In [45]:
# checks at the day of the week and the hour
feat_df['hour'] = feat_df['timestamp'].dt.hour
feat_df['dow'] = feat_df['timestamp'].dt.dayofweek
feat_df['is_weekend'] = feat_df['dow'].isin([5, 6]).astype(int)

# Filter to top categories only
temporal_df = feat_df[feat_df['category'].isin(top_categories)]

# Weekend ratio per user & category
weekend_ratio_cat = (
    temporal_df
    .groupby(['client_id', 'category'])['is_weekend']
    .mean()
    .unstack(fill_value=0)
    .rename(columns=lambda c: f'weekend_activity_ratio_cat_{int(c)}')
    .reset_index()
)



# Category Diversity Features 

In [48]:
# Distinct event types per category
event_type_diversity = (
    temporal_df
    .groupby(['client_id', 'category'])['event_type']
    .nunique()
    .unstack(fill_value=0)
    .rename(columns=lambda c: f'event_type_diversity_cat_{int(c)}')
    .reset_index()
)

# Sessions per category 
session_count_cat = (
    temporal_df
    .groupby(['client_id', 'category'])['timestamp']
    .nunique()
    .unstack(fill_value=0)
    .rename(columns=lambda c: f'session_count_cat_{int(c)}')
    .reset_index()
)


Merging the dataframes with the user_features 

In [51]:
user_features = (
    user_features
    .merge(monetary_cat_pivot, on='client_id', how='left')
    .merge(weekend_ratio_cat, on='client_id', how='left')
    .merge(event_type_diversity, on='client_id', how='left')
    .merge(session_count_cat, on='client_id', how='left')
    .fillna(0)
)


In [53]:
#user_features.columns.tolist()

In [55]:
# Compute recency per event_type and category
recency_df = (
    feat_df[feat_df['category'].isin(top_categories)]
    .groupby(['client_id', 'event_type', 'category'])['timestamp']
    .max()
    .reset_index()
)

# Calculate days since the last event prior to the cutoff date
recency_df['days_since'] = (cutoff_dt - recency_df['timestamp']).dt.days

# column per event_type-category pair
recency_pivot = recency_df.pivot(index='client_id', columns=['event_type', 'category'], values='days_since')

# Flatten the column names
recency_pivot.columns = [f'days_since_{evt}_cat_{int(cat)}' for evt, cat in recency_pivot.columns]
recency_pivot = recency_pivot.reset_index()

# Merge with existing user_features
user_features = user_features.merge(recency_pivot, on='client_id', how='left')

# Fill missing values 
user_features.fillna(0, inplace=True)


Merge the dataframes and check dimensions

In [58]:
# Group all purchases per user 
label_list = (
    label_buys
    .groupby('client_id')['category']
    .apply(list)
    .reset_index(name='label_categories')
)

# Filter each user's list to only include top categories
label_list['label_categories'] = label_list['label_categories'].apply(
    lambda cats: [c for c in cats if c in top_categories]
)

# Merge with user features
total_df = user_features.merge(label_list, on='client_id', how='left')

# Ensure label_categories is a list (even if empty)
total_df['label_categories'] = total_df['label_categories'].apply(
    lambda x: x if isinstance(x, list) else []
)

print("After merge, total_df shape:", total_df.shape)


After merge, total_df shape: (32663, 671)


# Implementation

ML classifiers

In [154]:
from sklearn.preprocessing import MultiLabelBinarizer


# split dataframe for training
X = total_df.drop(columns=['label_categories'])
y_raw = total_df['label_categories']

# Convert multi-label lists into binary matrix
mlb = MultiLabelBinarizer(classes=sorted(top_categories))
y = mlb.fit_transform(y_raw)

print(f"Total users: {X.shape[0]}")
print(f"Feature shape: {X.shape}, Label shape: {y.shape}")


Total users: 32663
Feature shape: (32663, 670), Label shape: (32663, 30)


In [156]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Initialize multi-label stratified splitter
msss = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)

# Perform the split
train_idx, test_idx = next(msss.split(X, y))


X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y[train_idx], y[test_idx]

# print shapes
print("Train users:", X_train.shape[0])
print("Test users:", X_test.shape[0])
print("Train label shape:", y_train.shape)
print("Test label shape:", y_test.shape)


Train users: 26130
Test users: 6533
Train label shape: (26130, 30)
Test label shape: (6533, 30)


In [158]:
#set up the parameters of the models that will be used

models = {
    'LightGBM': OneVsRestClassifier(
        LGBMClassifier(
            n_estimators=200,
            learning_rate=0.03,
            num_leaves=31,
            min_child_samples=50,
            subsample=0.7,
            colsample_bytree=0.7,
            n_jobs=-1,
            random_state=42
        )
    ),
    
    'XGBoost': OneVsRestClassifier(
        XGBClassifier(
            n_estimators=200,
            learning_rate=0.03,
            max_depth=6,
            min_child_weight=5,
            subsample=0.7,
            colsample_bytree=0.7,
            eval_metric='logloss',
            n_jobs=-1,
            random_state=42
        )
    )
}

Custom threshold for f1

In [195]:
'''
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

# === Optimize per-label thresholds for highest F1-score ===
def optimize_thresholds_for_f1(y_true, y_probs):
    thresholds = []
    for i in range(y_true.shape[1]):
        best_thresh = 0.5
        best_f1 = 0
        for t in np.linspace(0.05, 0.9, 18):  
            y_pred_col = (y_probs[:, i] >= t).astype(int)
            f1 = f1_score(y_true[:, i], y_pred_col, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = t
        thresholds.append(best_thresh)
    return thresholds

# === Data split ===
X_train_val, X_val, y_train_val, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# === Loop through models ===
for name, clf in models.items():
    print(f"\n=== {name} ===")
    clf.fit(X_train_val, y_train_val)  

    # Predict probabilities
    y_prob_val = clf.predict_proba(X_val)
    y_prob_test = clf.predict_proba(X_test)

    if isinstance(y_prob_val, list):
        y_prob_val = np.array([p[:, 1] for p in y_prob_val]).T
        y_prob_test = np.array([p[:, 1] for p in y_prob_test]).T

    #  Optimize thresholds for best F1
    thresholds = optimize_thresholds_for_f1(y_val, y_prob_val)

    # Predict test labels using optimized thresholds
    y_pred_test = np.array([
        (y_prob_test[:, i] >= thresholds[i]).astype(int)
        for i in range(y_prob_test.shape[1])
    ]).T

    # === Evaluation ===
    auroc = roc_auc_score(y_test, y_prob_test, average='macro')
    prec  = precision_score(y_test, y_pred_test, average='macro', zero_division=0)
    rec   = recall_score(y_test, y_pred_test, average='macro', zero_division=0)
    f1    = f1_score(y_test, y_pred_test, average='macro', zero_division=0)

    print(f"AUROC (macro):    {auroc:.4f}")
    print(f"Precision (macro):{prec:.4f}")
    print(f"Recall (macro):   {rec:.4f}")
    print(f"F1 (macro):       {f1:.4f}")
'''


=== LightGBM ===
[LightGBM] [Info] Number of positive: 206, number of negative: 20698
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012977 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11831
[LightGBM] [Info] Number of data points in the train set: 20904, number of used features: 354
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.009855 -> initscore=-4.609916
[LightGBM] [Info] Start training from score -4.609916
[LightGBM] [Info] Number of positive: 158, number of negative: 20746
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018208 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11831
[LightGBM] [Info] Number of data points in the train set: 20904, number of used features: 354


Custom threshold per label to increase precision

In [180]:
'''
from sklearn.model_selection import train_test_split

def optimize_thresholds_for_precision(y_true, y_probs):
    thresholds = []
    for i in range(y_true.shape[1]):
        best_thresh = 0.5
        best_prec = 0
        for t in np.linspace(0.05, 0.9, 18):  
            y_pred_col = (y_probs[:, i] >= t).astype(int)
            prec = precision_score(y_true[:, i], y_pred_col, zero_division=0)
            if prec > best_prec:
                best_prec = prec
                best_thresh = t
        thresholds.append(best_thresh)
    return thresholds

# Split off validation from training set
X_train_val, X_val, y_train_val, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# Loop through models
for name, clf in models.items():
    print(f"\n=== {name} ===")
    clf.fit(X_train_val, y_train_val)  

    # Predict probabilities
    y_prob_val = clf.predict_proba(X_val)
    y_prob_test = clf.predict_proba(X_test)

    if isinstance(y_prob_val, list):  # convert both val and test outputs
        y_prob_val = np.array([p[:, 1] for p in y_prob_val]).T
        y_prob_test = np.array([p[:, 1] for p in y_prob_test]).T

    # Optimize thresholds on validation set only
    thresholds = optimize_thresholds_for_precision(y_val, y_prob_val)

    # Predict test set labels using optimized thresholds
    y_pred_test = np.array([
        (y_prob_test[:, i] >= thresholds[i]).astype(int)
        for i in range(y_prob_test.shape[1])
    ]).T

    #  Final evaluation on test set
    auroc = roc_auc_score(y_test, y_prob_test, average='macro')
    prec  = precision_score(y_test, y_pred_test, average='macro', zero_division=0)
    rec   = recall_score(y_test, y_pred_test, average='macro', zero_division=0)
    f1    = f1_score(y_test, y_pred_test, average='macro', zero_division=0)

    print(f"AUROC (macro):    {auroc:.4f}")
    print(f"Precision (macro):{prec:.4f}")
    print(f"Recall (macro):   {rec:.4f}")
    print(f"F1 (macro):       {f1:.4f}")
'''


=== LightGBM ===
[LightGBM] [Info] Number of positive: 206, number of negative: 20698
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031904 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11831
[LightGBM] [Info] Number of data points in the train set: 20904, number of used features: 354
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.009855 -> initscore=-4.609916
[LightGBM] [Info] Start training from score -4.609916
[LightGBM] [Info] Number of positive: 158, number of negative: 20746
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028974 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11831
[LightGBM] [Info] Number of data points in the train set: 20904, number of used features: 354


Fixed threshold method

In [190]:

#Predict and evaluate with fixed threshold 0.5

#train the model and print macro averaged metrics
for name, clf in models.items():
    print(f"\n=== {name} ===")
    clf.fit(X_train, y_train)
    
    y_prob = clf.predict_proba(X_test)

    
    
    y_pred = (y_prob > 0.5).astype(int)

   
    auroc = roc_auc_score(y_test, y_prob, average='macro')
    prec  = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec   = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1    = f1_score(y_test, y_pred, average='macro', zero_division=0)
    
    


    print(f"AUROC (macro):    {auroc:.4f}")
    print(f"Precision (macro):{prec:.4f}")
    print(f"Recall (macro):   {rec:.4f}")
    print(f"F1 (macro):       {rec:.4f}")


=== LightGBM ===
[LightGBM] [Info] Number of positive: 265, number of negative: 25865
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025779 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13145
[LightGBM] [Info] Number of data points in the train set: 26130, number of used features: 356
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.010142 -> initscore=-4.580916
[LightGBM] [Info] Start training from score -4.580916
[LightGBM] [Info] Number of positive: 194, number of negative: 25936
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022070 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13145
[LightGBM] [Info] Number of data points in the train set: 26130, number of used features: 356


In [203]:

#Predict and evaluate with fixed threshold 0.005

for name, clf in models.items():
    print(f"\n=== {name} ===")
    clf.fit(X_train, y_train)
    
    y_prob = clf.predict_proba(X_test)

    
    
    y_pred = (y_prob > 0.005).astype(int)

   
    auroc = roc_auc_score(y_test, y_prob, average='macro')
    prec  = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec   = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1    = f1_score(y_test, y_pred, average='macro', zero_division=0)
    
    


    print(f"AUROC (macro):    {auroc:.4f}")
    print(f"Precision (macro):{prec:.4f}")
    print(f"Recall (macro):   {rec:.4f}")
    print(f"F1 (macro):       {rec:.4f}")


=== LightGBM ===
[LightGBM] [Info] Number of positive: 265, number of negative: 25865
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014438 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13145
[LightGBM] [Info] Number of data points in the train set: 26130, number of used features: 356
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.010142 -> initscore=-4.580916
[LightGBM] [Info] Start training from score -4.580916
[LightGBM] [Info] Number of positive: 194, number of negative: 25936
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015132 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13145
[LightGBM] [Info] Number of data points in the train set: 26130, number of used features: 356
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.007424 -> initscore=-4.895529
[LightGBM] [Info] Start training from score -4.8

# TOP K CATEGORY PREDICTION 

In [164]:
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

#set up a function to evaluate the model on topK suggestions using macro averaged precision recall f1 and auroc
def evaluate_topk_model(model, model_name, X_test, y_test, mlb, top_k=1):
    #Predict probabilities 
    y_prob = model.predict_proba(X_test)

    # Get true labels and class names 
    true_label_sets = [set(labels) for labels in mlb.inverse_transform(y_test)]
    category_labels = mlb.classes_

    #  Get top-k predicted classes for each user 
    top_k_indices = np.argsort(y_prob, axis=1)[:, -top_k:][:, ::-1]
    top_k_pred_sets = [
        set(category_labels[i] for i in row)
        for row in top_k_indices
    ]

    #  Evaluate Recall@k, Precision@k, F1@k 
    recall_at_k = []
    precision_at_k = []
    f1_at_k = []

    for true, pred in zip(true_label_sets, top_k_pred_sets):
        true_positives = len(true & pred)
        recall = true_positives / len(true) if true else 0.0
        precision = true_positives / top_k
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0 else 0.0
        )

        recall_at_k.append(recall)
        precision_at_k.append(precision)
        f1_at_k.append(f1)

    y_pred_bin = np.zeros_like(y_test)
    for i, preds in enumerate(top_k_indices):
        y_pred_bin[i, preds] = 1
    
    #  Aggregate metrics int macro averages
    avg_precision_macro = precision_score(y_test, y_pred_bin, average='macro', zero_division=0)
    avg_recall_macro = recall_score(y_test, y_pred_bin, average='macro', zero_division=0)
    avg_f1_macro = f1_score(y_test, y_pred_bin, average='macro', zero_division=0)
    macro_auroc = roc_auc_score(y_test, y_prob, average='macro')

    #  Output results 
    print(f"\n=== Top-{top_k} Evaluation for {model_name} ===")
    print(f"Macro AUROC:       {macro_auroc:.4f}")
    print(f"Macro Precision@{top_k}: {avg_precision_macro:.4f}")
    print(f"Macro Recall@{top_k}:    {avg_recall_macro:.4f}")
    print(f"Macro F1@{top_k}:        {avg_f1_macro:.4f}")

    # return full DataFrame 
    return pd.DataFrame({
        'client_id': X_test['client_id'].values,
        'true_categories': [list(labels) for labels in true_label_sets],
        f'top_{top_k}_predicted': [list(pred) for pred in top_k_pred_sets],
        'recall_at_k': recall_at_k,
        'precision_at_k': precision_at_k,
        'f1_at_k': f1_at_k
    })


In [166]:
# Evaluation at topK = 10
for name, model in models.items():
    model.fit(X_train, y_train)  
    results_topK = evaluate_topk_model(model, name, X_test, y_test, mlb, top_k=10)


[LightGBM] [Info] Number of positive: 265, number of negative: 25865
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030229 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13145
[LightGBM] [Info] Number of data points in the train set: 26130, number of used features: 356
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.010142 -> initscore=-4.580916
[LightGBM] [Info] Start training from score -4.580916
[LightGBM] [Info] Number of positive: 194, number of negative: 25936
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.031864 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13145
[LightGBM] [Info] Number of data points in the train set: 26130, number of used features: 356
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.007424 -> initscore=-4.895529
[L

In [207]:
# Evaluation at topK = 3
for name, model in models.items():
    model.fit(X_train, y_train)  
    results_topK = evaluate_topk_model(model, name, X_test, y_test, mlb, top_k=3)


[LightGBM] [Info] Number of positive: 265, number of negative: 25865
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014482 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13145
[LightGBM] [Info] Number of data points in the train set: 26130, number of used features: 356
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.010142 -> initscore=-4.580916
[LightGBM] [Info] Start training from score -4.580916
[LightGBM] [Info] Number of positive: 194, number of negative: 25936
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023453 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13145
[LightGBM] [Info] Number of data points in the train set: 26130, number of used features: 356
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.007424 -> initscore=-4.895529
[L

# Hyperparameter tuning

In [114]:
#Tune hyperparameters
from sklearn.model_selection import RandomizedSearchCV
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# 5-fold multi label 
cv = MultilabelStratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# LightGBM hyperparameter space
lgb_base = OneVsRestClassifier(
    LGBMClassifier(random_state=42),
    n_jobs=-1
)
lgb_param_dist = {
    'estimator__n_estimators':    [100, 200],
    'estimator__learning_rate':   [0.01, 0.05],
    'estimator__num_leaves':      [31, 50],
    'estimator__min_child_samples': [5, 10, 20],
    'estimator__subsample':       [0.6, 0.8, 1.0],
    'estimator__colsample_bytree': [0.6, 0.8, 1.0],
}




In [116]:
# Randomized search for LightGBM
lgb_search = RandomizedSearchCV(
    lgb_base,
    lgb_param_dist,
    n_iter=10,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2
)
lgb_search.fit(X_train, y_train)

print("Best LightGBM params:", lgb_search.best_params_)
best_lgb = lgb_search.best_estimator_

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best LightGBM params: {'estimator__subsample': 0.8, 'estimator__num_leaves': 31, 'estimator__n_estimators': 100, 'estimator__min_child_samples': 5, 'estimator__learning_rate': 0.01, 'estimator__colsample_bytree': 0.8}


Save the best parameters

In [126]:
from joblib import dump
dump(best_lgb, "best_lgb_model.pkl")

['best_lgb_model.pkl']

In [128]:
# Tune hyperparameters of xgboost classifier 
xgb_base = OneVsRestClassifier(
    XGBClassifier(
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    ),
    n_jobs=-1
)

# Xgb  hyperparameter search space
xgb_param_dist = {
    'estimator__n_estimators':    [100, 200],
    'estimator__learning_rate':   [0.01, 0.05],
    'estimator__max_depth':       [3, 6],
    'estimator__subsample':       [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0],
}

In [130]:
# Search for the best hyperparameters
from sklearn.model_selection import RandomizedSearchCV
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

cv = MultilabelStratifiedKFold(n_splits=3, shuffle=True, random_state=42)

xgb_search = RandomizedSearchCV(
    xgb_base,
    xgb_param_dist,
    n_iter=10,
    scoring='roc_auc',
    cv=cv,                
    random_state=42,
    n_jobs=-1,
    verbose=2
)

# train
xgb_search.fit(X_train, y_train)

# Inspect best parameters
print("Best XGBoost params:", xgb_search.best_params_)
best_xgb = xgb_search.best_estimator_

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best XGBoost params: {'estimator__subsample': 0.8, 'estimator__n_estimators': 100, 'estimator__max_depth': 3, 'estimator__learning_rate': 0.01, 'estimator__colsample_bytree': 0.8}


save the best xgb parameters

In [132]:
dump(best_xgb, "best_xgb_model.pkl")

['best_xgb_model.pkl']

# importing saved model parameters

In [168]:
from joblib import load

best_lgb = load("best_lgb_model.pkl")
best_xgb = load("best_xgb_model.pkl")

# Evaluating the tuned models 

Custom thresholds

In [170]:

'''
def evaluate_with_thresholds(clf, X, y_true, thresholds=None, default_threshold=0.5, name=None):
    # Predict probabilities
    y_prob = clf.predict_proba(X)

    # Convert to array 
    if isinstance(y_prob, list):
        y_prob = np.array([p[:, 1] for p in y_prob]).T

    # Use default threshold if none provided
    if thresholds is None:
        thresholds = [default_threshold] * y_prob.shape[1]

    # Apply per-label thresholds
    y_pred = np.array([
        (y_prob[:, i] >= thresholds[i]).astype(int)
        for i in range(y_prob.shape[1])
    ]).T

    # Evaluate
    auroc = roc_auc_score(y_true, y_prob, average='macro')
    prec  = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec   = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1    = f1_score(y_true, y_pred, average='macro', zero_division=0)
    hloss = hamming_loss(y_true, y_pred)

    if name:
        model_name = name
    elif hasattr(clf, 'estimator'):
        model_name = f"OneVsRest({clf.estimator.__class__.__name__})"
    else:
        model_name = clf.__class__.__name__
    print(f"\n=== {model_name} Evaluation ===")
    print(f"AUROC (macro):    {auroc:.4f}")
    print(f"Precision (macro):{prec:.4f}")
    print(f"Recall (macro):   {rec:.4f}")
    print(f"F1 Score (macro): {f1:.4f}")

X_val = pd.DataFrame(X_val, columns=X_train.columns)
X_test = pd.DataFrame(X_test, columns=X_train.columns)

# Step 1: Tune thresholds on validation set (held-out from training)
thresholds_lgb = optimize_thresholds_for_precision(y_val, best_lgb.predict_proba(X_val))
thresholds_xgb = optimize_thresholds_for_precision(y_val, best_xgb.predict_proba(X_val))

# Step 2: Evaluate with those thresholds on the unseen test set
evaluate_with_thresholds(best_lgb, X_test, y_test, thresholds=thresholds_lgb)
evaluate_with_thresholds(best_xgb, X_test, y_test, thresholds=thresholds_xgb)
'''


=== OneVsRest(LGBMClassifier) Evaluation ===
AUROC (macro):    0.6707
Precision (macro):0.3034
Recall (macro):   0.0396
F1 Score (macro): 0.0686

=== OneVsRest(XGBClassifier) Evaluation ===
AUROC (macro):    0.6776
Precision (macro):0.2910
Recall (macro):   0.0240
F1 Score (macro): 0.0394


In [138]:
# function to evaluate for default threshold of 0.5
def evaluate(clf, X_test, y_test, threshold=0.5):
    y_prob = clf.predict_proba(X_test)
    y_pred = (y_prob > threshold).astype(int)
    print(f"\n=== {clf.__class__.__name__} @ thresh={threshold} ===")
    print("AUROC (macro):      ", roc_auc_score(y_test, y_prob, average='macro'))
    print("Precision (mac):    ", precision_score(y_test, y_pred, average='macro', zero_division=0))
    print("Recall (macro):     ", recall_score(y_test, y_pred, average='macro', zero_division=0))
    print("F1 (macro):         ", f1_score(y_test, y_pred, average='macro', zero_division=0))
    

evaluate(best_lgb, X_test, y_test)
evaluate(best_xgb, X_test, y_test)


=== OneVsRestClassifier @ thresh=0.05 ===
AUROC (macro):       0.670674033804731
Precision (mac):     0.1739537795020052
Recall (macro):      0.2184414482246177
F1 (macro):          0.1893273613111492

=== OneVsRestClassifier @ thresh=0.05 ===
AUROC (macro):       0.677611307763666
Precision (mac):     0.1763344339650996
Recall (macro):      0.21982117966904355
F1 (macro):          0.19065579492973134


# TOP K CATEGORIES SUGGESTION

In [172]:
#evaluate topK = 5 method after tuning the hyperparameters
results_lgb = evaluate_topk_model(best_lgb, "LightGBM", X_test, y_test, mlb, top_k=10)
results_xgb = evaluate_topk_model(best_xgb, "XGBoost", X_test, y_test, mlb, top_k=10)



=== Top-10 Evaluation for LightGBM ===
Macro AUROC:       0.6707
Macro Precision@10: 0.0314
Macro Recall@10:    0.5140
Macro F1@10:        0.0561

=== Top-10 Evaluation for XGBoost ===
Macro AUROC:       0.6776
Macro Precision@10: 0.0367
Macro Recall@10:    0.5116
Macro F1@10:        0.0628


# Neural Network approach 

In [174]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Add, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# === Preprocess ===
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
X_tr, X_val, y_tr, y_val = train_test_split(X_train_s, y_train, test_size=0.2, random_state=42)

# === Model architecture ===
n_features = X_tr.shape[1]
n_labels = y_tr.shape[1]

inputs = Input(shape=(n_features,))
x = Dense(512)(inputs)
x = BatchNormalization()(x)
x = LeakyReLU()(x)
x = Dropout(0.4)(x)

x1 = Dense(256)(x)
x1 = BatchNormalization()(x1)
x1 = LeakyReLU()(x1)
x1 = Dropout(0.3)(x1)

x2 = Dense(256)(x1)
x2 = BatchNormalization()(x2)
x2 = LeakyReLU()(x2)
x2 = Dropout(0.3)(x2)

x = Add()([x1, x2])  # residual

x = Dense(128)(x)
x = BatchNormalization()(x)
x = LeakyReLU()(x)
x = Dropout(0.2)(x)

outputs = Dense(n_labels, activation='sigmoid')(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=BinaryCrossentropy(label_smoothing=0.05),
    metrics=['AUC']
)

# === Callbacks ===
early_stop = EarlyStopping(monitor='val_AUC', mode='max', patience=10, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_AUC', mode='max', factor=0.5, patience=5, min_lr=1e-5)

# === Training ===
epochs = 50
batch_size = 128

model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stop, reduce_lr],
    verbose=1  
)


Epoch 1/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 12s 34ms/step - AUC: 0.5442 - loss: 0.4114 - val_AUC: 0.5634 - val_loss: 0.1740 - learning_rate: 0.0010
Epoch 2/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - AUC: 0.6226 - loss: 0.1569 - val_AUC: 0.6665 - val_loss: 0.1499 - learning_rate: 0.0010
Epoch 3/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - AUC: 0.6480 - loss: 0.1545 - val_AUC: 0.6997 - val_loss: 0.1494 - learning_rate: 0.0010
Epoch 4/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - AUC: 0.6690 - loss: 0.1536 - val_AUC: 0.7116 - val_loss: 0.1486 - learning_rate: 0.0010
Epoch 5/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - AUC: 0.6772 - loss: 0.1528 - val_AUC: 0.7158 - val_loss: 0.1487 - learning_rate: 0.0010
Epoch 6/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - AUC: 0.6833 - loss: 0.1521 - val_AUC: 0.7217 - val_loss: 0.1483 - learning_rate: 0.0010
Epoch 7/50
164/164 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - AUC: 0.6940 - loss: 0.1515 - val_AUC: 0.7194 - val_loss: 0.1487 - learning_rate: 0.001

Predicting with topK 

In [176]:
from sklearn.metrics import roc_auc_score
import numpy as np

# === Predict probabilities ===
y_prob_test = model.predict(X_test_s, verbose=0)

# === Top-k setup ===
top_k = 10
category_labels = mlb.classes_ 

# === Get top-k predicted label sets ===
top_k_indices = np.argsort(y_prob_test, axis=1)[:, -top_k:][:, ::-1]
top_k_pred_sets = [set(category_labels[i] for i in row) for row in top_k_indices]

# === True label sets ===
true_label_sets = [set(category_labels[i] for i in np.where(row == 1)[0]) for row in y_test]

# === Compute Top-k Recall, Precision, F1 ===
recall_at_k = []
precision_at_k = []
f1_at_k = []

for true, pred in zip(true_label_sets, top_k_pred_sets):
    tp = len(true & pred)
    recall = tp / len(true) if true else 0.0
    precision = tp / top_k
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    recall_at_k.append(recall)
    precision_at_k.append(precision)
    f1_at_k.append(f1)

macro_recall = recall_score(true_label_sets, top_k_pred_sets, average='macro', zero_division=0)
macro_precision = precision_score(true_label_sets, top_k_pred_sets, average='macro', zero_division=0)
macro_f1 = f1_score(true_label_sets, top_k_pred_sets, average='macro', zero_division=0)
macro_auroc = roc_auc_score(y_test, y_prob_test, average='macro')

# === Output results ===
print("\n=== Final Evaluation on Test Set using the NN ===")
print(f"Macro AUROC:         {macro_auroc:.4f}")
print(f"Average Precision@{top_k}: {macro_precision:.4f}")
print(f"Average Recall@{top_k}:    {macro_recall:.4f}")
print(f"Average F1@{top_k}:        {macro_f1:.4f}")



=== Final Evaluation on Test Set using the NN ===
Macro AUROC:         0.6743
Average Precision@10: 0.0205
Average Recall@10:    0.1747
Average F1@10:        0.0361
